# 25: Manuscript figures for the RF drift cell chapter

**What this is.** This notebook renders the figure set for the drift cell chapter from the banked campaign file that notebook 24 writes (`data/g1m_dt0.2ns.json`), with fresh solves of the same deck supplying the field predictions. Nothing here flies the campaign. The only flights are the three example trajectories of the device panel, and re-running this notebook after any future campaign rebuilds every figure from the new bank.

**The figures.**

* **F0** shows the simulated device, its solved potential, and example trajectories.
* **F1** plots transmission against RF amplitude at the three drift fields.
* **F2** presents the arrival-time distributions at the lowest drift field, which move to later times as the amplitude rises.
* **F3** sets the mean arrival times at all three fields against the first-order collision-theory drift times.
* **F4** plots the axial force at three radii. The force oscillates with the ring pitch and does not reverse at any radius the flown ions reached.
* **F4s** (supporting information) gives the predicted slowdown at each radius from the drift-diffusion calculation, with the flown occupancy and the onset of potential minima marked.
* **F5** compares the flown delays with the parameter-free prediction, both relative to the 50 Vpp condition.
* **F6** compares the pitch-crossing time distributions with the uniform drift-diffusion distribution. The slow tail grows modestly and stays continuous.

In [ ]:
# ---- import-origin guard (run me FIRST) --------------------------------
# THIS notebook belongs to a repo; it must run against THAT repo's
# ion_gym, not whatever `import ion_gym` happens to find (a pip-installed
# copy, or an old tree on PYTHONPATH). A version mismatch does not fail
# politely -- it surfaces mid-run as a confusing AttributeError on some
# API the stale copy predates. So: locate the repo from this notebook's
# working directory, put it FIRST on sys.path, then verify the imported
# package actually came from here -- and REFUSE with the remedy if not.
# GENERATED CELL: every notebook carries one identical copy, stamped from
# a single definition in the development tree. Edits made here are
# overwritten the next time the notebook is regenerated.
import sys
from pathlib import Path

# leading underscores here are DELIBERATE (stated reason): this
# cell is stamped into every notebook and must not collide with or
# pollute the study's own names
_here = Path.cwd().resolve()
ROOT_GUARD = next((p for p in (_here, *_here.parents)
                   if (p / "ion_gym").is_dir() and (p / "notebooks").is_dir()),
                  None)
if ROOT_GUARD is None:
    raise RuntimeError(
        f"cannot locate the ion_gym repo at or above {_here}; start the "
        f"kernel in the repo root or in a folder inside it")
if str(ROOT_GUARD) not in sys.path:
    sys.path.insert(0, str(ROOT_GUARD))

import ion_gym
_origin = Path(ion_gym.__file__).resolve().parent
if _origin.parent != ROOT_GUARD:
    raise RuntimeError(
        f"imported ion_gym v{ion_gym.__version__} from {_origin}, which is "
        f"NOT this repo ({ROOT_GUARD / 'ion_gym'}). Either the kernel "
        f"already imported a stale copy (restart the kernel and run this "
        f"cell first) or another copy shadows the repo (a pip-installed "
        f"ion_gym: `pip uninstall ion_gym`; or a stale PYTHONPATH entry). "
        f"Refusing now beats an AttributeError several cells later.")
print(f"ion_gym v{ion_gym.__version__} · loaded from {_origin}")


<!-- origin-guard-note -->
The cell above only pins this notebook to its own repo's `ion_gym`. The notebook proper begins below.


In [ ]:
# ---- session resources: throughput choices, not physics -----------------
# The same defaults the workbench offers beside its Fly button. A deck
# never declares these, and a seeded flight returns the same results at
# any worker count.
from ion_gym.physics.ensemble_driver import default_workers, fly_all
from ion_gym.physics.sizing import default_record_budget_gb, resources_line

N_WORKERS = default_workers()      # ion-flight threads: every logical
                                   # core but two, so the kernel and the
                                   # OS stay responsive
RECORD_BUDGET_GB = default_record_budget_gb()   # GB the trajectory record
                                   # may use: RECORD_REFUSE_FRACTION of RAM,
                                   # the workbench default. build_run warns
                                   # at half of it and refuses above it.
                                   # None = RAM unreadable, fixed limits.
print(resources_line(N_WORKERS, RECORD_BUDGET_GB))

## 0. Parameters — every choice, named

In [ ]:
# every choice this notebook makes, named here; the physics settings
# themselves come from the BANK, so a figure can never disagree with
# the flights it draws
import json
import math
import time
from pathlib import Path

import numpy as np
from IPython.display import Image, display
from scipy.interpolate import CubicSpline

from ion_gym.io.paths import examples_dir, outputs_dir, repo_root
from ion_gym.physics import ensemble_driver as ed
from ion_gym.physics.constants import E_CHG, KG_AMU, K_B_EV
from ion_gym.progress import quote, track
from ion_gym.viz.viz_core import apply_house_style, DEFAULT_DPI
from ion_gym.viz.design_maps import ProfileCurve, render_profiles

# the shipped manuscript_examples folder holds this notebook and the deck
# generator; the bank notebook 24 flies and these figures live in the
# outputs folder
STUDY = repo_root() / 'manuscript_examples' / 'drift_cell'
OUT = outputs_dir('manuscript_examples', 'drift_cell', 'manuscript_figures')
BANK_FILE = outputs_dir('manuscript_examples', 'drift_cell', 'data',
                        'g1m_dt0.2ns.json')   # the flown campaign

SHOW_EP = 0.35          # field of the single-field figures F2, F4, F6: the headline (lowest) field
BASELINE_VPP = 50       # F5 reference: the lowest amplitude with full transmission (no survivor bias)
N_EXAMPLE = 3           # example trajectories in the device panel
N_FINE = 4096           # fine points per pitch in the washboard integrals
N_FORCE = 512           # spline samples per pitch in the F4 force curves (the lattice has 12)
OCC_PCT = (5.0, 95.0)   # occupancy band percentiles shown on F4s
R_GRID_MAX_MM = 2.25    # predictor radius limit: 0.25 mm inside the ring bore
MAX_BEYOND_GRID_FRACTION = 0.01   # F5 refuses if more crossings than this fall past the grid
GAS_DAMPED = True       # scale the pseudopotential by (w tau)^2/(1+(w tau)^2), tau = mK/q

KT_EV = K_B_EV * 298.0
C_EP = {1.51: '#0072B2', 0.507: '#009E73', 0.35: '#D55E00'}   # Okabe-Ito, one color per field
C_V = {0: '#999999', 15: '#56B4E9', 50: '#009E73', 100: '#E69F00', 200: '#D55E00'}
apply_house_style()
print(f'bank: {BANK_FILE.name} | figures -> {OUT}')

# ---- per-figure publication styling -----------------------------------
# One entry per figure holds everything adjustable for publication: the
# title, the figure size, the legend placement, and the four font sizes.
# Edit the entry and re-run that figure's cell. Titles use {} fields that
# the figure cells fill from the live values. legend_loc takes any
# matplotlib location name ('upper left', 'lower right', 'best', ...) or
# an (x, y) tuple in axes fractions anchoring the legend's lower-left
# corner, for placing the legend in a gap no named location reaches.
# F0 renders through the multi-axis scene renderer, which solves its own
# panel sizes from the geometry and draws no legend, so its entry
# carries the font sizes and an overall scale factor instead of figsize
# and legend keys.
FIG_STYLE = {
    'F0': dict(title=('F0  SRIG drift cell: {n_rings} rings, {bore:g} mm '
                      'i.d., E/P {ep:g}, {vpp:g} Vpp @ {mhz:g} MHz, '
                      '{torr:g} Torr {gas}, {n_ex} example trajectories'),
               scale=1.0, title_fontsize=11, label_fontsize=11,
               tick_fontsize=10),
    'F1': dict(title='F1  RF confinement restores transmission at reduced '
                     'drift fields',
               # title NOT drawn (show_titles=False in the cell): the
               # document caption carries it, per the
               # review; fonts raised to match F3
               figsize=(7.0, 4.6), legend_loc='lower right', legend_ncol=1,
               legend_fontsize=11, title_fontsize=9, label_fontsize=13,
               tick_fontsize=12),
    'F2': dict(title='F2  Arrival times move later as RF amplitude rises '
                     '(E/P {ep:g} V/cm/Torr)',
               # title NOT drawn (show_titles=False in the cell): the
               # document caption carries it, per the
               # review; fonts raised to match F3
               figsize=(7.0, 4.6), legend_loc='upper left', legend_ncol=1,
               legend_fontsize=11, title_fontsize=9, label_fontsize=13,
               tick_fontsize=12),
    'F3': dict(title='F3  Mean arrival times rise with RF amplitude, most '
                     'at the lowest drift field',
               # the title is NOT drawn (show_titles=False in the cell):
               # the document caption carries it, per the
               # review; fonts raised the same day
               figsize=(7.0, 4.6), legend_loc=(0.04, 0.15), legend_ncol=1,
               legend_fontsize=11, title_fontsize=9, label_fontsize=13,
               tick_fontsize=12),
    'F4': dict(title='F4  The axial force oscillates with the ring pitch '
                     'and does not reverse at radii the ions reached '
                     '({vpp:g} Vpp, E/P {ep:g})',
               # the title is NOT drawn (show_titles=False in the cell):
               # the document caption carries it, per the
               # review; the legend sits top center in the
               # gap between the force peaks, with headroom added in
               # the cell; fonts raised to match F3
               figsize=(7.0, 4.6), legend_loc='upper center', legend_ncol=1,
               legend_fontsize=11, title_fontsize=9, label_fontsize=13,
               tick_fontsize=12),
    'F4s': dict(title='F4s  Predicted slowdown at each radius, with the '
                      'flown occupancy and the onset of potential minima '
                      '(E/P {ep:g})',
                figsize=(7.0, 4.6), legend_loc='upper left', legend_ncol=1,
                legend_fontsize=9, title_fontsize=9, label_fontsize=11,
                tick_fontsize=10),
    'F5': dict(title='F5  Flown delays compared with the '
                     'effective-potential prediction, relative to 50 Vpp',
               # the title is NOT drawn (show_titles=False in the cell):
               # the document caption carries it, per the
               # review; the baseline hmark enters the legend
               # (hmark_legend) so its label cannot sit on the curves
               # that converge at zero; fonts raised to match F3
               figsize=(7.0, 4.6), legend_loc='upper left', legend_ncol=1,
               legend_fontsize=10, title_fontsize=9, label_fontsize=13,
               tick_fontsize=12),
    'F6': dict(title='F6  Pitch crossing times against uniform '
                     'drift-diffusion (E/P {ep:g})',
               # title NOT drawn (show_titles=False in the cell): the
               # document caption carries it, per the
               # review; fonts raised to match F3
               figsize=(7.0, 4.6), legend_loc='lower left', legend_ncol=1,
               legend_fontsize=11, title_fontsize=9, label_fontsize=13,
               tick_fontsize=12),
}


def fig_kwargs(key):
    # the style entry minus the title, ready to unpack into the renderer
    st = dict(FIG_STYLE[key])
    st.pop('title')
    return st

## 1. The banked campaign

The bank is the campaign file notebook 24 writes: one record per (E/P, Vpp) cell carrying every setting it was flown with, the per-ion transits, and the residence time and mean radius of every ring-pitch crossing. The cell below refuses a missing bank and refuses a bank whose cells disagree on their settings, then reconstructs the device description FROM the bank, so every solve and every figure label in this notebook describes exactly what flew.

In [ ]:
# load the bank, verify internal consistency, and take the device from it
import sys

sys.path.insert(0, str(STUDY))
from make_rf_drift_cell import rf_drift_cell_spec, PITCH_MM   # noqa: E402

if not BANK_FILE.exists():
    raise RuntimeError(
        f'no campaign bank at {BANK_FILE}: fly it with notebook 24 '
        f'(24_srig_rf_mechanism.ipynb, RUN=True) first; this notebook '
        f'only renders what that one banked')
cells = json.load(open(BANK_FILE))
SETTINGS = ('bore_id_mm', 'n_rings', 'rf_freq_hz', 'p_torr', 'T_k', 'gas',
            'ion_mass_da', 'ion_charge', 'sigma_m2', 'source_ring',
            'exit_ring', 'dt_ns', 'seed', 'plane_a_ring', 'plane_b_ring')
first = cells[0]
for c in cells[1:]:
    diffs = [k for k in SETTINGS if c.get(k) != first.get(k)]
    if diffs:
        raise RuntimeError(
            f'bank cells disagree on {diffs} (E/P {c["e_over_p"]:g}, '
            f'{c["vpp"]:g} Vpp vs the first cell): one campaign, one '
            f'operating point -- do not mix banks')
DEVICE = {k: first[k] for k in ('bore_id_mm', 'n_rings', 'rf_freq_hz',
                                'p_torr', 'T_k', 'gas', 'ion_mass_da',
                                'ion_charge', 'sigma_m2', 'source_ring',
                                'exit_ring')}
DT_NS = first['dt_ns']
SEED = first['seed']
L_MM = first['L_mm']
BY = {(c['e_over_p'], c['vpp']): c for c in cells}
EPS = sorted({c['e_over_p'] for c in cells}, reverse=True)
VPPS = sorted({c['vpp'] for c in cells})
OP = (f"Allen&Bush cell i.d. {DEVICE['bore_id_mm']:g} mm, "
      f"{DEVICE['n_rings']}-ring model, planes {first['x_a_mm']:.1f}->"
      f"{first['x_b_mm']:.1f} mm ({first['plane_b_ring'] - first['plane_a_ring']} "
      f"pitches, {L_MM:g} mm), {DEVICE['rf_freq_hz'] / 1e6:g} MHz, "
      f"{DEVICE['p_torr']:g} Torr {DEVICE['gas']} {DEVICE['T_k']:g} K, "
      f"GRGDS2+ {DEVICE['ion_mass_da']:g} Da z=+{DEVICE['ion_charge']}, "
      f"HS sigma {DEVICE['sigma_m2']:.3g} m^2, dt {DT_NS:g} ns")
print(f'{len(cells)} cells banked; {OP}')
print('E/P    Vpp  births  transits  mean transit (us)')
for c in sorted(cells, key=lambda c: (-c['e_over_p'], c['vpp'])):
    t = (f"{c['transit_mean_us']:9.1f} +- {c['transit_se_us']:.1f}"
         if c.get('n_transit', 0) else '        no transits')
    print(f"{c['e_over_p']:5} {c['vpp']:4g} {c['n_ions']:6d} "
          f"{c.get('n_transit', 0):8d} {t}")


def crossings(c):
    # every recorded pitch-crossing mean radius of a cell, flattened
    return np.concatenate([np.asarray(x, float)
                           for x in c['r_period_per_ion'] if len(x)])

## 2. F0: the simulated device

F0 renders the instrument the bank describes from the solver's own electrode mask and overlays example trajectories at the lowest drift field and the full amplitude. These are the only flights in this notebook, and their measured time is quoted before they run.

In [ ]:
# the device panel: one solve plus N_EXAMPLE example ions
from ion_gym.physics.sim_build import build_run
from ion_gym.viz import viz_core as V

spec_geo = rf_drift_cell_spec(**DEVICE, rf_vpp=max(VPPS),
                              e_over_p=SHOW_EP, n_ions=N_EXAMPLE,
                              dt_ns=DT_NS, seed=SEED)
t_build = time.time()
model_geo, fly_geo, cols_geo, births_geo = build_run(spec_geo, record_budget_gb=RECORD_BUDGET_GB)
quote('F0 example flights after a measured build', N_EXAMPLE,
      per_item_s=(time.time() - t_build))
trajs = []
fates = []
for tr, summ in fly_all(N_EXAMPLE, fly_geo, n_workers=N_WORKERS):
    trajs.append(tr)
    fates.append(summ['kind'])
st0 = FIG_STYLE['F0']
sc = V.scene_from_simspec(
    spec_geo, model_geo, field='phi', trajs=trajs, fates=fates,
    title=st0['title'].format(
        n_rings=DEVICE['n_rings'], bore=DEVICE['bore_id_mm'], ep=SHOW_EP,
        vpp=max(VPPS), mhz=DEVICE['rf_freq_hz'] / 1e6,
        torr=DEVICE['p_torr'], gas=DEVICE['gas'], n_ex=N_EXAMPLE))
# the scene renderer reads font sizes from matplotlib rcParams, so the
# F0 entry is applied through a temporary rc context around the call
import matplotlib as mpl
with mpl.rc_context({'axes.labelsize': st0['label_fontsize'],
                     'axes.titlesize': st0['title_fontsize'],
                     'xtick.labelsize': st0['tick_fontsize'],
                     'ytick.labelsize': st0['tick_fontsize']}):
    fig0 = V.render_mpl(sc)
# panel sizes are solved from the geometry, so overall size is adjusted
# by the scale factor rather than a fixed figsize
w0, h0 = fig0.get_size_inches()
fig0.set_size_inches(w0 * st0['scale'], h0 * st0['scale'])
png = OUT / 'F0_device.png'
fig0.savefig(png, dpi=DEFAULT_DPI)
display(Image(str(png)))

## 3. The folded effective potential and the drift-speed calculation

The prediction rests on pure functions of the solved potential, shared by F4, F4s, and F5. The effective potential at each radius is folded over the measured pitches into one periodic profile, and the exact mean drift speed of an overdamped ion in that tilted periodic potential (Stratonovich) gives the predicted slowdown. A flat profile must return zero, and that control is printed. The force curves of F4 come from the periodic spline of the same folded profile, so their smoothness is interpolation of the lattice profile, stated on the figure.

In [ ]:
# fold, spline, washboard, and the gas factor; flat-control printed
from ion_gym.physics.ccs import low_field_mobility_m2Vs


def fold_pitch_profile(z_mm, u_ev, z_a_mm, z_b_mm, pitch_mm):
    
    """Fold a 1-D potential over whole pitches between two planes.
    Returns the pitch-phase grid (mm), the folded ripple with the linear
    ramp removed (eV), the ramp force F = -dU/dz over the window
    (eV/mm), and the largest pitch-to-pitch deviation in SHAPE from the
    folded mean (eV, each pitch compared after removing its own mean)."""

    h = float(z_mm[1] - z_mm[0])
    ia = int(round((z_a_mm - z_mm[0]) / h))
    ib = int(round((z_b_mm - z_mm[0]) / h))
    n_per = int(round(pitch_mm / h))
    if abs(n_per * h - pitch_mm) > 1e-9 or (ib - ia) % n_per:
        raise ValueError(
            f'window {z_a_mm}->{z_b_mm} mm is not a whole number of '
            f'{pitch_mm} mm pitches on the {h} mm lattice')
    span = z_mm[ib] - z_mm[ia]
    F = (u_ev[ia] - u_ev[ib]) / span
    seg = u_ev[ia:ib] + F * (z_mm[ia:ib] - z_mm[ia])
    stack = seg.reshape(-1, n_per)
    folded = stack.mean(axis=0)
    shape = stack - stack.mean(axis=1, keepdims=True)
    spread = float(np.abs(shape - (folded - folded.mean())).max())
    phase = np.arange(n_per) * h
    return phase, folded, float(F), spread


def periodic_spline(phase_mm, p_ev, pitch_mm):
    """Periodic cubic spline of a folded profile."""
    return CubicSpline(np.append(phase_mm, pitch_mm),
                       np.append(p_ev, p_ev[0]), bc_type='periodic')


def washboard_slowdown(p_fine_ev, pitch_mm, F_ev_mm, kT_ev):
    """t/t_free - 1 for overdamped drift + diffusion in p(x) - F x with p
    periodic over one pitch (Stratonovich mean velocity; the mobility
    cancels in the ratio). A flat p returns 0 to rounding."""
    n = p_fine_ev.size
    d = pitch_mm / n
    b = 1.0 / kT_ev
    p = p_fine_ev - p_fine_ev.min()
    c = b * F_ev_mm
    kern = np.exp(-c * np.arange(n) * d) * (-np.expm1(-c * d)) / c
    inner = np.real(np.fft.ifft(np.fft.fft(np.exp(-b * p)) * np.fft.fft(kern)))
    denom = d * np.sum(np.exp(b * p) * inner)
    free = pitch_mm * (-np.expm1(-c * pitch_mm)) / c
    return float(denom / free - 1.0)


# gas factor from the same cross section the kernel flies
spec_k = rf_drift_cell_spec(**DEVICE, rf_vpp=0.0, e_over_p=1.0, n_ions=1)
# the ion's MASS in Da (review C-11: the function takes mass for every
# charge; passing m/z gave a lighter reduced mass)
K_M2VS = low_field_mobility_m2Vs(
    spec_k.collisions, DEVICE['ion_mass_da'], DEVICE['ion_charge'])
TAU_S = (DEVICE['ion_mass_da'] * KG_AMU * K_M2VS
         / (DEVICE['ion_charge'] * E_CHG))
WT = 2 * np.pi * DEVICE['rf_freq_hz'] * TAU_S
GAMMA = WT ** 2 / (1 + WT ** 2) if GAS_DAMPED else 1.0
Z_A = first['x_a_mm']
Z_B = first['x_b_mm']
print(f'momentum relaxation tau {TAU_S * 1e6:.3f} us, gas factor {GAMMA:.4f}')
print('flat-profile control (must be ~0):',
      washboard_slowdown(np.zeros(N_FINE), PITCH_MM, 0.35, KT_EV))


# effective potential per (E/P, Vpp), solves shared in-process
MODELS = {}


def model(ep, vpp):
    if (ep, vpp) not in MODELS:
        sp = rf_drift_cell_spec(**DEVICE, rf_vpp=float(vpp), e_over_p=ep,
                                n_ions=1, dt_ns=DT_NS, seed=SEED)
        MODELS[(ep, vpp)] = build_run(sp, record_budget_gb=RECORD_BUDGET_GB)[0]
    return MODELS[(ep, vpp)]


def u_surface(ep, vpp):
    z, rf, pe_dc, em = model(ep, 0).pe_surface(
        mz=DEVICE['ion_mass_da'], charge=DEVICE['ion_charge'])
    z2, rf2, pe_rf, em2 = model(ep, vpp).pe_surface(
        mz=DEVICE['ion_mass_da'], charge=DEVICE['ion_charge'])
    return z, rf, pe_dc + GAMMA * (pe_rf - pe_dc)

## 4. F1: transmission against RF amplitude

F1 plots the fraction of born ions that cross the full measured span, in absolute percent. Without RF only a few percent survive the two low fields. From 50 Vpp upward every born ion crosses, which is why the 50 Vpp condition is the reference for F5.

In [ ]:
# F1: transmission against amplitude, one curve per field
curves = []
for ep in EPS:
    x = [v for v in VPPS if (ep, v) in BY]
    y = [100.0 * BY[(ep, v)]['n_transit'] / BY[(ep, v)]['n_ions'] for v in x]
    curves.append(ProfileCurve(f'E/P {ep:g} V/cm/Torr', x, y,
                               {'marker': 'o', 'color': C_EP[ep]}))
fig = render_profiles(curves, xlabel='RF Amplitude (Vpp)',
                      ylabel='Transmission (%)',
                      title=FIG_STYLE['F1']['title'],
                      subtitle=OP,
                      show_titles=False,   # the document caption carries the title
                      **fig_kwargs('F1'))
png = OUT / 'F1_transmission.png'
fig.savefig(png, dpi=DEFAULT_DPI)
display(Image(str(png)))

## 5. F2: arrival-time distributions at the lowest drift field

F2 presents the cumulative arrival-time curves at the lowest drift field, one color per amplitude, with no normalization. The shift of the curves to later times with amplitude is the observation the chapter explains. The RF-off condition is excluded with its survivor count printed, because almost no ion crosses the span without confinement.

In [ ]:
# F2: arrival ECDFs at SHOW_EP, absolute milliseconds
curves = []
for v in VPPS:
    c = BY[(SHOW_EP, v)]
    t = np.sort(np.asarray(c['transit_us_all'])) / 1e3
    if t.size < 50:
        print(f'{v:g} Vpp excluded: {t.size} survivors of '
              f'{c["n_ions"]} births without confinement')
        continue
    frac = np.arange(1, t.size + 1) / t.size
    print(f'{v:4g} Vpp: mean {t.mean():.3f} ms, n {t.size}')
    curves.append(ProfileCurve(f'{v:g} Vpp', t, frac,
                               {'drawstyle': 'steps-post', 'color': C_V[v]}))
fig = render_profiles(curves,
                      xlabel='Arrival Time (ms)',
                      ylabel='Cumulative Fraction of Ions',
                      title=FIG_STYLE['F2']['title'].format(ep=SHOW_EP),
                      subtitle=OP, hmarks=[(0.5, 'median')],
                      show_titles=False,   # the document caption carries the title
                      **fig_kwargs('F2'))
png = OUT / 'F2_arrivals_ecdf.png'
fig.savefig(png, dpi=DEFAULT_DPI)
display(Image(str(png)))

## 6. F3: mean arrival times against collision-theory drift

F3 compares the mean arrival times at all three fields with the first-order Mason-Schamp drift time of the same cross section, drawn as a dotted line per field. Every curve sits about 2 percent below its line, an offset between the hard-sphere collision model and the first-order theory that the cell prints and the figure subtitle states, so the gap cannot be misread as an RF effect. The RF effect is the rise of each curve with amplitude. Readers are cautioned against reading the proximity of the low-amplitude points to the dotted lines as absolute accuracy, since the model offset and the RF delay carry opposite signs and partially cancel near full amplitude. The accuracy statement that survives both offsets is relative, the drift time added between low and full amplitude, which F5 quantifies.

In [ ]:
# F3: mean arrival vs amplitude with the Mason-Schamp lines
t_ms_ideal = {ep: L_MM / (K_M2VS * ep * DEVICE['p_torr'] / 10.0)
              for ep in EPS}
ref_hi = BY[(max(EPS), 0)]
print(f"kernel vs Mason-Schamp at E/P {max(EPS):g}, RF off: "
      f"{ref_hi['transit_mean_us']:.1f} vs {t_ms_ideal[max(EPS)]:.1f} us "
      f"({ref_hi['transit_mean_us'] / t_ms_ideal[max(EPS)] - 1:+.2%})")
curves = []
hmarks = []
for ep in EPS:
    x = [v for v in VPPS if BY[(ep, v)].get('n_transit', 0) >= 50]
    y = [BY[(ep, v)]['transit_mean_us'] / 1e3 for v in x]
    e = [BY[(ep, v)]['transit_se_us'] / 1e3 for v in x]
    curves.append(ProfileCurve(f'E/P {ep:g} V/cm/Torr', x, y,
                               {'marker': 'o', 'color': C_EP[ep],
                                'markerfacecolor': 'none'}, yerr=e))
    # one legend entry covers all three identical reference lines
    hmarks.append((t_ms_ideal[ep] / 1e3,
                   'Mason-Schamp ideal' if ep == EPS[0] else ''))
fig = render_profiles(
    curves, xlabel='RF Amplitude (Vpp)',
    ylabel='Mean Arrival Time (ms)',
    title=FIG_STYLE['F3']['title'],
    subtitle=OP + '; dotted lines = first-order Mason-Schamp drift, same '
                  'cross section; the ~2% offset of every curve from its '
                  'line is the hard-sphere kernel against that first-order '
                  'theory, not an RF effect',
    hmarks=hmarks, hmark_legend=True, show_titles=False,
    **fig_kwargs('F3'))
png = OUT / 'F3_arrival_time.png'
fig.savefig(png, dpi=DEFAULT_DPI)
display(Image(str(png)))

## 7. F4: the axial force at three radii

**Where the solid lines come from.** Each curve is computed from the solved fields, with no flown data in the curve itself. The field solve supplies the DC potential and the RF field amplitude at every lattice node, and the effective potential U(z, r) is the DC potential energy plus the damped pseudopotential (section 3). At one fixed radius, the axial profile of U between the measurement planes is folded onto a single ring pitch after removing the linear ramp, and the plotted force is the ramp value minus the slope of the folded profile, evaluated from a periodic cubic spline at N_FORCE points per pitch and tiled over two pitches for display. The lattice supplies 12 points per pitch, so the spline is interpolation for smoothness and adds no physics, and the largest pitch-to-pitch deviation in the shape of the folded profile is printed by the cell. The flown campaign enters only through the choice of radii: the median radius of the recorded pitch crossings, the farthest radius any recorded crossing reached, and a third radius computed from the field as the first lattice radius at which the force reverses.

**What the reader should see.** All three curves oscillate with the 1.5 mm ring period about the dotted line marking the uniform ladder field, and the pitch average of each curve equals that ladder value, so the applied gradient is delivered in full on average and the rings redistribute the force along each pitch without changing its mean. The swing grows with radius because the pseudopotential gradient grows toward the ring electrodes. At the cloud median the force stays between roughly 175 and 315 meV/mm, and at the farthest recorded crossing its minimum is still near 95 meV/mm, so every ion in the campaign always felt a forward force. Only the third curve, at a radius beyond every recorded crossing, touches the second dotted line at zero, which is the threshold at which the effective potential first forms a local minimum and holding an ion becomes possible.

**Why an uneven force delays arrival.** An ion moves slowly through the half of each pitch where the force is reduced and quickly through the half where the force is raised, and because transit time depends on the average of the reciprocal speed, the slow half adds more time than the fast half removes. That imbalance, summed over the 33 measured pitches, is the delay quantified in F5, and the absence of any zero crossing at occupied radii is the geometric statement that the delay arises without trapping.

In [ ]:
# F4: force curves at the cloud median, the farthest crossing, and the
# first push-reversal radius, all from the solved field at SHOW_EP
c_show = BY[(SHOW_EP, max(VPPS))]
r_occ = crossings(c_show)
r_med = float(np.median(r_occ))
r_far = float(r_occ.max())
z_f, rf_f, u_f = u_surface(SHOW_EP, max(VPPS))
x_fine = np.arange(2 * N_FORCE) * (PITCH_MM / N_FORCE)


def force_at(r_want):
    # axial force (meV/mm) over two pitches at the nearest lattice radius
    j = int(np.argmin(np.abs(rf_f - r_want)))
    ph, p, F, spread = fold_pitch_profile(z_f, u_f[:, j], Z_A, Z_B, PITCH_MM)
    spl = periodic_spline(ph, p, PITCH_MM)
    force = (F - spl(x_fine % PITCH_MM, 1)) * 1e3
    return float(rf_f[j]), force, F, spread


# first lattice radius where the force reverses: min force <= 0
r_rev = None
spread_max = 0.0
for j in np.nonzero((rf_f >= 0) & (rf_f <= R_GRID_MAX_MM))[0]:
    ph, p, F, spread = fold_pitch_profile(z_f, u_f[:, j], Z_A, Z_B, PITCH_MM)
    spread_max = max(spread_max, spread)
    if (F - periodic_spline(ph, p, PITCH_MM)(x_fine % PITCH_MM, 1)).min() <= 0:
        r_rev = float(rf_f[j])
        break
if r_rev is None:
    raise RuntimeError(
        f'no push-reversal radius found inside {R_GRID_MAX_MM} mm at '
        f'{max(VPPS):g} Vpp: the F4 story needs re-examination, refuse '
        f'rather than drawing a figure without its threshold')
print(f'cloud median {r_med:.2f} mm, farthest crossing {r_far:.2f} mm, '
      f'force first reverses at {r_rev:.2f} mm; largest pitch-to-pitch '
      f'shape deviation {spread_max * 1e3:.2f} meV')
ramp_mev_mm = (DEVICE['ion_charge'] * SHOW_EP * DEVICE['p_torr'] / 10.0) * 1e3
curves = []
for r_want, lab, col in (
        (r_med, f'cloud median ({r_med:.2f} mm)', '#0072B2'),
        (r_far, f'farthest crossing ({r_far:.2f} mm)', '#E69F00'),
        (r_rev, f'first reversal ({r_rev:.2f} mm)', '#D55E00')):
    r_got, force, F, spread = force_at(r_want)
    curves.append(ProfileCurve(lab, x_fine, force, {'color': col}))
fig = render_profiles(
    curves, xlabel='Axial Distance (mm)',
    ylabel='Axial Force (meV/mm)',
    title=FIG_STYLE['F4']['title'].format(vpp=max(VPPS), ep=SHOW_EP),
    subtitle=OP + '; force = -dU/dz of the effective potential, '
                  'pitch-folded over the measured window and splined; '
                  'a potential minimum requires the force to reach zero',
    hmarks=[(ramp_mev_mm, 'uniform ladder field'),
            (0.0, 'zero axial force')],
    show_titles=False,   # the document caption carries the title
    **fig_kwargs('F4'))
ax = fig.axes[0]
y0, y1 = ax.get_ylim()
# headroom so the one-row legend sits above the force peaks instead of
# on them; the legend location is axes-relative, so it rides up too
ax.set_ylim(y0, y1 + 0.30 * (y1 - y0))
png = OUT / 'F4_force.png'
fig.savefig(png, dpi=DEFAULT_DPI)
display(Image(str(png)))

## 8. F4s: predicted slowdown at each radius (supporting information)

F4s shows the predicted fractional slowdown at each radius, computed from the drift-diffusion mean speed over the folded potential, for every banked amplitude at the lowest drift field. F4s is the quantitative form of F4 and supplies the prediction tested in F5. The shaded band spans the radii of 90 percent of the flown pitch crossings at full amplitude, and the marks give the farthest crossing and the onset of potential minima. The run time of the prediction loop is measured on its first condition and quoted before the loop runs.

In [ ]:
# F4s: s(r) per cell via the washboard result; also fills PRED for F5
def predict(ep, vpp):
    z, rf, u = u_surface(ep, vpp)
    keep = (rf >= -1e-12) & (rf <= R_GRID_MAX_MM + 1e-9)
    radii = rf[keep]
    s = np.zeros(radii.size)
    well_r = None
    for k, j in enumerate(np.nonzero(keep)[0]):
        ph, p, F, spread = fold_pitch_profile(z, u[:, j], Z_A, Z_B, PITCH_MM)
        spl = periodic_spline(ph, p, PITCH_MM)
        x1 = np.arange(N_FINE) * (PITCH_MM / N_FINE)
        s[k] = washboard_slowdown(spl(x1), PITCH_MM, F, KT_EV)
        if well_r is None and spl(x1, 1).max() > F:
            well_r = float(radii[k])
    return dict(r=radii, s=s, well_r=well_r)


todo = [(ep, v) for ep in EPS for v in VPPS if v > 0]
t_first = time.time()
PRED = {todo[0]: predict(*todo[0])}
quote('field predictions per cell (first cell measured; the solves are '
      'shared, so the rest re-weight)', len(todo) - 1,
      per_item_s=(time.time() - t_first))
for key in track(todo[1:], label='prediction cells'):
    PRED[key] = predict(*key)

r05, r95 = np.percentile(r_occ, OCC_PCT)
curves = []
for v in (v for v in VPPS if v > 0):
    pr = PRED[(SHOW_EP, v)]
    sel = (pr['r'] <= 1.45) & (pr['s'] * 100 > 1e-4)
    curves.append(ProfileCurve(f'{v:g} Vpp', pr['r'][sel],
                               pr['s'][sel] * 100,
                               {'marker': '.', 'color': C_V[v]}))
well200 = PRED[(SHOW_EP, max(VPPS))]['well_r']
fig = render_profiles(
    curves, xlabel='Radius (mm)',
    ylabel='Predicted Slowdown (%)', logy=True,
    title=FIG_STYLE['F4s']['title'].format(ep=SHOW_EP),
    subtitle=OP + f'; shaded span = where {OCC_PCT[0]:g}-{OCC_PCT[1]:g}% '
                  f'of flown pitch crossings sit at {max(VPPS):g} Vpp',
    vmarks=[(r05, f'cloud {OCC_PCT[0]:g}%'), (r95, f'cloud {OCC_PCT[1]:g}%'),
            (r_far, 'farthest crossing'),
            (well200, f'potential minima begin ({max(VPPS):g} Vpp)')],
    **fig_kwargs('F4s'))
ax = fig.axes[0]
ax.axvspan(r05, r95, color=C_V[max(VPPS)], alpha=0.12, zorder=0)
png = OUT / 'F4s_slowdown_vs_radius.png'
fig.savefig(png, dpi=DEFAULT_DPI)
display(Image(str(png)))

## 9. F5: flown delays against the prediction

F5 compares the flown delays with the prediction, both expressed as drift time added relative to the 50 Vpp condition, with standard errors on the flown points and one color per field. The prediction is the F4s slowdown of each condition averaged over that condition's own recorded crossing radii, then referenced to the 50 Vpp condition the same way, with no adjusted quantity anywhere in the chain. The 50 Vpp reference matches the form of the published shifts, and because every condition at or above 50 Vpp transmits every born ion, the reference also removes the survivor-selection bias that conditions with transmission loss carry. Crossings past the predictor grid are excluded from the average as a printed, counted skip, refused above a named fraction. Such crossings occur only in unconfined conditions, where the prediction is flat near zero.

In [ ]:
# F5: flown vs predicted shifts, both referenced to BASELINE_VPP
def crossing_weighted(c, pr):
    # mean predicted slowdown over the cell's own crossings
    r = crossings(c)
    beyond = r > pr['r'].max()
    frac = float(beyond.mean())
    if frac > MAX_BEYOND_GRID_FRACTION:
        raise ValueError(
            f'E/P {c["e_over_p"]:g}, {c["vpp"]:g} Vpp: {int(beyond.sum())} '
            f'of {r.size} crossings ({frac:.1%}) past the predictor grid '
            f'at {pr["r"].max():.2f} mm: too many to exclude without '
            f'moving the average')
    if beyond.any():
        print(f'E/P {c["e_over_p"]:g}, {c["vpp"]:g} Vpp: excluded '
              f'{int(beyond.sum())} of {r.size} crossings ({frac:.3%}) '
              f'past the predictor grid')
    kept = r[~beyond]
    return float(np.interp(kept, pr['r'], pr['s']).mean())


curves = []
print('E/P    Vpp   flown shift         predicted')
for ep in EPS:
    base = BY[(ep, BASELINE_VPP)]
    if base['n_transit'] < base['n_ions']:
        print(f'E/P {ep:g}: baseline {BASELINE_VPP} Vpp transmits '
              f'{base["n_transit"]}/{base["n_ions"]} -- survivor bias '
              f'enters this family, read it with that caveat')
    cw_base = crossing_weighted(base, PRED[(ep, BASELINE_VPP)])
    xs = []
    flown = []
    err = []
    predicted = []
    for v in (v for v in VPPS if v > BASELINE_VPP):
        c = BY[(ep, v)]
        shift = c['transit_mean_us'] / base['transit_mean_us'] - 1
        se = (1 + shift) * math.hypot(
            c['transit_se_us'] / c['transit_mean_us'],
            base['transit_se_us'] / base['transit_mean_us'])
        pred = (1 + crossing_weighted(c, PRED[(ep, v)])) / (1 + cw_base) - 1
        xs.append(v)
        flown.append(shift * 100)
        err.append(se * 100)
        predicted.append(pred * 100)
        print(f'{ep:5} {v:4g}  {shift * 100:+7.3f} +- {se * 100:.3f} %   '
              f'{pred * 100:+7.3f} %')
    curves.append(ProfileCurve(f'flown, E/P {ep:g}', xs, flown,
                               {'marker': 'o', 'ls': 'none',
                                'color': C_EP[ep]}, yerr=err))
    curves.append(ProfileCurve(f'predicted, E/P {ep:g}',
                               [BASELINE_VPP] + xs, [0.0] + predicted,
                               {'ls': '--', 'marker': '.',
                                'color': C_EP[ep]}))
fig = render_profiles(
    curves, xlabel='RF Amplitude (Vpp)',
    ylabel='Added Drift Time (%)',
    title=FIG_STYLE['F5']['title'],
    subtitle=OP + f'; shifts relative to the {BASELINE_VPP} Vpp '
                  f'condition; prediction = drift-diffusion over the '
                  f'effective potential, averaged over the flown occupancy',
    hmarks=[(0.0, f'{BASELINE_VPP} Vpp baseline')],
    hmark_legend=True,   # the baseline label rides the legend, not the plot
    show_titles=False,   # the document caption carries the title
    **fig_kwargs('F5'))
png = OUT / 'F5_pred_vs_flown.png'
fig.savefig(png, dpi=DEFAULT_DPI)
display(Image(str(png)))

## 10. F6: pitch-crossing statistics against uniform drift-diffusion

The quantity on the horizontal axis is a first-passage time: the time an ion takes to move from one pitch plane to the plane one ring pitch (1.5 mm) downstream, with the crossing instants interpolated between recorded steps. Every transit contributes 33 of these times, so the conditions shown carry tens of thousands of crossings each, and the exact counts are printed by the cell.

The vertical axis is the survival function, the fraction of crossings slower than t, on a logarithmic scale. The question under test concerns rare slow crossings, since a trapped subpopulation would appear as a small excess of very slow events. A histogram compresses that tail into a few sparsely filled bins, while the survival function needs no binning, and the frequency of any slowness threshold reads directly off the curve (e.g., the t at which the curve passes 10⁻³ is the crossing time exceeded once per thousand crossings).

The uniform drift-diffusion distribution is a zero-parameter null hypothesis. For a particle undergoing nothing but steady drift and diffusion, the first-passage time across a fixed distance L follows the inverse-Gaussian distribution with mean L/v_d and shape L²/2D exactly. Both parameters are fixed from outside the plotted curve, v_d from that condition's own measured mean crossing time and D from the Einstein relation at the gas temperature, and the measured quantity and the uniform drift-diffusion distribution describe the same first-passage experiment. At low amplitude the measured curve lies on the uniform drift-diffusion curve, which certifies the transport statistics of the simulation. At full amplitude the tail rises modestly above the uniform drift-diffusion curve but stays continuous to the last recorded crossing, an excess consistent with a diffusion coefficient slightly above the gas-temperature value, as weak micromotion heating produces. A trapped population would instead appear as a shoulder or a second component, and none appears. The bulk of each curve is decimated for file size and the extreme tail is kept whole, since the tail is the quantity under test.

In [ ]:
# F6: residence survival against uniform drift-diffusion, absolute microseconds
from math import erf, exp, sqrt


def ig_cdf(x, mu, lam):
    # inverse-Gaussian CDF (drifted-Brownian first passage)
    Phi = lambda u: 0.5 * (1 + erf(u / sqrt(2)))
    a = sqrt(lam / x)
    return Phi(a * (x / mu - 1)) + exp(2 * lam / mu) * Phi(-a * (x / mu + 1))


pitch_mm = L_MM / (first['plane_b_ring'] - first['plane_a_ring'])
curves = []
for v in (min(v for v in VPPS if v > 0), max(VPPS)):
    c = BY[(SHOW_EP, v)]
    taus = [np.asarray(t) for t in c['tau_us_per_ion'] if len(t)]
    n_per = min(len(t) for t in taus)
    tau = np.array([t[:n_per] for t in taus]).ravel()
    mu = tau.mean()
    D = KT_EV * (pitch_mm / mu) / (DEVICE['ion_charge'] * c['E_ideal_v_mm'])
    lam = pitch_mm ** 2 / (2 * D)
    u = np.sort(tau)
    surv = 1.0 - np.arange(1, u.size + 1) / u.size
    stride = max(1, u.size // 4000)
    keep = np.zeros(u.size, bool)
    keep[::stride] = True
    keep[-300:] = True
    print(f'{v:4g} Vpp: {tau.size} crossings recorded')
    curves.append(ProfileCurve(
        f'{v:g} Vpp simulated', u[keep],
        np.maximum(surv[keep], 0.5 / u.size),
        {'drawstyle': 'steps-post', 'color': C_V[v]}))
    tt = np.linspace(mu * 0.4, u.max() * 1.02, 400)
    s_ref = np.array([1.0 - ig_cdf(x, mu, lam) for x in tt])
    curves.append(ProfileCurve(
        f'{v:g} Vpp uniform drift-diffusion', tt,
        np.maximum(s_ref, 1e-6),
        {'ls': '--', 'color': C_V[v], 'lw': 1.0}))
fig = render_profiles(
    curves,
    xlabel='Crossing Time, t (µs)',
    ylabel='P(Crossing Time > t)', logy=True,
    title=FIG_STYLE['F6']['title'].format(ep=SHOW_EP),
    subtitle=OP + f"; one ring pitch = {pitch_mm:g} mm; uniform drift-diffusion = "
                  f"inverse-Gaussian first passage at each "
                  f"amplitude's own measured drift speed with "
                  f"gas-temperature diffusion, no free parameter",
    show_titles=False,   # the document caption carries the title
    **fig_kwargs('F6'))
png = OUT / 'F6_residence.png'
fig.savefig(png, dpi=DEFAULT_DPI)
display(Image(str(png)))

## Read-out

**The order of the argument.** F1 establishes that confinement enables transmission at reduced drift fields. F2 presents the arrival-time observation in the units an instrument reports. F3 adds the quantitative delays and the collision-theory baseline. F4 gives the mechanism in the axial force, and F5 tests that mechanism against the flown delays. F6 excludes a trapped population. F4s carries the quantitative chain behind F4 and F5 and belongs in supporting information.

**What the alternative outcomes would have looked like.** If path length caused the delay, the three curves of F3 would rise by the same fraction at every field, and they do not. If trapping caused the delay, the full-amplitude tail of F6 would develop a shoulder above the uniform drift-diffusion curve and the flown radii of F4 would reach the zero-force region, and neither happens. If the effective-potential picture were wrong, the dashed predictions of F5 would miss the measured shifts entirely rather than carrying most of each shift.

**Stated caveats, on the figures and here.** The 2 percent offset in F3 separates the hard-sphere collision model from first-order Mason-Schamp theory and is printed by that cell. Conditions below full transmission carry survivor-selection bias, which is why F5 is referenced to the 50 Vpp condition. The F5 residual, under one percentage point and roughly independent of drift field, is consistent with weak collisional micromotion heating and is reported, not attributed. The smooth curves of F4 are the periodic spline of the pitch-folded lattice profile, and the largest pitch-to-pitch deviation in the shape of the folded profile is printed by that cell.

**Styling.** The title, legend placement, and font sizes for each figure are set in the FIG_STYLE block of the parameter cell. Changing an entry and re-running the one figure cell regenerates that figure. The renderer (design_maps.render_profiles) carries these options with defaults that reproduce its previous output, so no other caller changes.

**Housekeeping.** The fold and washboard functions here duplicate notebook 24's inlined copies. One sentence proposal: move both into a shared study module in a chartered change so the two notebooks cannot drift, and stop there pending a ruling.